# Implementación aproximación mediante JTBD

## Importación de librerias necesarias


In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")


JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [3]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T, DataFrame
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.ml.feature import VectorAssembler

In [4]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('modeling_jtbd')
spark = spark_utils.spark

2025-10-19 17:19:36.669918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-19 17:19:36.682061: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-19 17:19:36.685517: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-19 17:19:36.695242: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-19 17:19:38.694503: W tensorflow/compiler/tf2

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-39c776fc-6b18-4fc1-9d72-c3112d21da57;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 105ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

## Cargar datos fuente de entrenamiento

In [5]:
GOLD_ENCODING = 'gold.encoding'
GOLD_PREMODELING = 'gold.premodeling'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
SCHEMA = 'silver.preprocess'

In [28]:
sample_equitative_hierarchical = spark.read.format('delta').load(spark_utils.path(
    'sample_equitative_hierarchical', catalog=GOLD_SCHEMA_CLUSTER
))

main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = GOLD_PREMODELING
))

reviews_indexed = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed', catalog = SCHEMA
))

meta_items_features_text_clean_embeddings = spark.read.format('parquet').load(spark_utils.path(
    'meta_items_features_text_clean_embeddings', 
    catalog = GOLD_ENCODING
))

meta_items_features_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_text_clean', catalog = GOLD_PREMODELING
))


In [29]:
meta_items_features_text_clean.show(1)

+-----------+--------------------+--------------+---------------+---------+
|parent_asin|    feature_sentence|feature_number|sentence_number|record_id|
+-----------+--------------------+--------------+---------------+---------+
| 0110400550|Fully compatible ...|             1|              1|        1|
+-----------+--------------------+--------------+---------------+---------+
only showing top 1 row



In [7]:
REGENERATE_INTERMEDIATE_TABLES = False

In [20]:
products_average_rating = (
    reviews_indexed
    .groupBy('parent_asin')
    .agg(
        F.avg('rating').alias('average_rating'),
    )
)

In [22]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        products_average_rating
            .write
            .format('delta')
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'products_average_rating', catalog = GOLD_PREMODELING
            ))
    )

products_average_rating = spark.read.format('delta').load(spark_utils.path(
    'products_average_rating', catalog = GOLD_PREMODELING
))

## Construir datasets para inferencia

### Construir datasets de productos y reseñas para predicción de calificación

In [34]:
meta_items_features_average_rating = (
    meta_items_features_text_clean.alias('A').join(
        meta_items_features_text_clean_embeddings.alias('B'),
        on = 'record_id',
        how = 'inner'
    ).join(
        products_average_rating.alias('C'),
        on = 'parent_asin',
        how = 'inner'
    ).join(
        sample_equitative_hierarchical.alias('D'),
        on = 'parent_asin',
        how = 'inner'
    ).select(
        'A.parent_asin',
        'B.text_embeddings',
        'C.average_rating',
        'D.cluster_hierarchical',
    )
)

In [ ]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        meta_items_features_average_rating
            .write
            .format('delta')
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_features_average_rating', catalog = GOLD_PREMODELING
            ))
    )

meta_items_features_average_rating = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_average_rating', catalog = GOLD_PREMODELING
))

In [36]:
meta_items_features_average_rating.schema.fields

[StructField('parent_asin', StringType(), True),
 StructField('text_embeddings', ArrayType(DoubleType(), True), True),
 StructField('average_rating', DoubleType(), True),
 StructField('cluster_hierarchical', LongType(), True)]

In [37]:
@F.pandas_udf(T.DoubleType())
def cosine_similarity(vec1, vec2):
    return vec1.combine(vec2, lambda a, b: float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))))

In [38]:
gamma = 5.0

In [52]:
def calculate_jtbd_inspired_rating(
    self_features,
    base_features_ratings
):
    df_cosine_simmilarity_individual = (
        self_features.alias('A').join(
            base_features_ratings.alias('B'),
            on = 'cluster_hierarchical',
            how = 'inner'
        ).select(
            cosine_similarity(
                F.col('A.text_embeddings'),
                F.col('B.text_embeddings')
            ).alias('cosine_similarity'),
            F.col('B.average_rating').alias('average_rating')
        )
    )

    print(df_cosine_simmilarity_individual.show())

    df_cosine_simmilarity_individual = (
        df_cosine_simmilarity_individual
            .withColumn(
                "final_score",
                F.col("cosine_similarity") *
                F.exp(-gamma * F.pow(F.col("cosine_similarity") - (F.col("average_rating") / 5.0), 2))
            )
            .select(
                F.avg('final_score').alias('jtbd_inspired_rating')
            )
    )
    result = df_cosine_simmilarity_individual.collect()[0]['jtbd_inspired_rating']
    return result / 2 + 0.5
    

In [53]:
jtbd_inspired = calculate_jtbd_inspired_rating(
    (
        meta_items_features_average_rating
            .filter(F.col('parent_asin') == 'B00005LEOS')
            .select(
                'text_embeddings',
                'average_rating',
                'cluster_hierarchical'
            )
    ),
    meta_items_features_average_rating.select(
        'text_embeddings',
        'average_rating',
        'cluster_hierarchical'
    )
)


+--------------------+------------------+
|   cosine_similarity|    average_rating|
+--------------------+------------------+
| 0.01036523954151043| 4.666666666666667|
| 0.16931468633929014|3.5350877192982457|
|-0.04521915292345...|2.7142857142857144|
| 0.05841246772951892|3.8666666666666667|
|-0.00481802692318...|2.6666666666666665|
|0.047400137027064145| 3.581081081081081|
| 0.12839275655436322|3.6666666666666665|
|-0.04543422549365855|               3.5|
|  0.1644813307954381|3.7142857142857144|
|0.007350711525267783|4.4423076923076925|
|-0.05925390205214885|3.7333333333333334|
|-0.02022361377598...|3.9166666666666665|
| 0.09541657273398024| 4.901639344262295|
| 0.10615716518917394| 4.136363636363637|
|0.001610815945091035|               3.5|
|0.017102328980590404|3.7333333333333334|
|-0.02628670924412275|               3.5|
|-0.03184471215609...| 4.450704225352113|
| 0.11105515561733821|3.7333333333333334|
|-0.01345549269457...| 4.090909090909091|
+--------------------+------------

In [54]:
jtbd_inspired

0.510316391670183

In [9]:
products_data_cross_joined = (
    tmp_products_data.alias('A').join(
        tmp_products_data.alias('B'),
        on = 'cluster_hierarchical',
        how = 'inner'
    ).select(
        F.col('A.parent_asin').alias('parent_asin_1'),
        F.col('A.parent_asin').alias('parent_asin_2'),
        F.col('A.cluster_hierarchical').alias('cluster_hierarchical_1'),
        F.col('A.cluster_hierarchical').alias('cluster_hierarchical_2'),
        F.col('A.features').alias('features_1'),
        F.col('A.features').alias('features_2'),
        *[  
            F.col(f'A.{col}').alias(f'{col}_1')
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ],
        *[
            F.col(f'B.{col}').alias(f'{col}_2')
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ]
        
    )
)

In [13]:
if REGENERATE_INTERMEDIATE_TABLES:
    (
        products_data_cross_joined
            .write
            .format('delta')
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'products_data_cross_joined', catalog = GOLD_PREMODELING
            ))
    )

products_data_cross_joined = spark.read.format('delta').load(spark_utils.path(
    'products_data_cross_joined', catalog = GOLD_PREMODELING
))